In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FileFormatsDemo") \
    .master("local[*]") \
    .getOrCreate()
print("SparkSession started successfully...")

In [ ]:
df = spark.read.json("customers.json")

df.show()
df.printSchema()

In [ ]:
from pyspark.sql.functions import col

df.filter(
    col("Amount") > 50000
).show()

In [ ]:
from pyspark.sql.functions import sum, avg

df.groupBy("City").agg(
    sum("Amount").alias("TotalAmount"),
    avg("Amount").alias("AverageAmount")
).show()

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Amount", DoubleType(), True)
])

df = spark.read \
    .schema(schema) \
    .json("customers.json")

df.printSchema()

In [ ]:
employee_df = spark.read.json(
    "employees_data.json"
)

employee_df.printSchema()

In [ ]:
employee_df.select(
    "EmployeeID",
    "Name",
    "Address.City",
    "Address.State"
).show()

In [ ]:
flat_df = employee_df.select(
    col("EmployeeID"),
    col("Name"),
    col("Department"),
    col("Address.City").alias("City"),
    col("Address.State").alias("State")
)

flat_df.show()

In [ ]:
flat_df.write \
    .mode("overwrite") \
    .json("output/employees_json")

In [ ]:
df = spark.read.json(
    "customers.json"
)

df.write \
    .mode("overwrite") \
    .parquet("output/customers_parquet")

In [ ]:
df = spark.read.parquet(
    "output/customers_parquet"
)

df.show()
df.printSchema()

In [ ]:
df = spark.read.parquet(
    "output/customers_parquet"
)

result = df.select(
    "CustomerID",
    "City",
    "Amount"
)

result.show()

In [ ]:
result = df.filter(
    col("Amount") > 50000
).select(
    "Name",
    "City",
    "Amount"
)

result.show()

In [ ]:
data = [
    (101, "Amit", "Sales", "Mumbai", 2025, 55000, 250000),
    (102, "Priya", "Sales", "Pune", 2025, 58000, 310000),
    (103, "Rahul", "IT", "Mumbai", 2025, 75000, 0),
    (104, "Sneha", "IT", "Pune", 2025, 82000, 0),
    (105, "Karan", "HR", "Delhi", 2025, 50000, 0),
    (106, "Neha", "HR", "Mumbai", 2025, 52000, 0),

    (107, "Arjun", "Sales", "Delhi", 2026, 62000, 380000),
    (108, "Pooja", "Sales", "Mumbai", 2026, 65000, 420000),
    (109, "Rohan", "IT", "Pune", 2026, 85000, 0),
    (110, "Meera", "IT", "Delhi", 2026, 88000, 0),
    (111, "Vikram", "HR", "Pune", 2026, 55000, 0),
    (112, "Anjali", "HR", "Delhi", 2026, 57000, 0)
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "city",
    "year",
    "salary",
    "sales"
]

df = spark.createDataFrame(data, columns)

df.show()

In [ ]:
df.write \
    .mode("overwrite") \
    .partitionBy("department", "year") \
    .parquet("employee_partitioned")

In [ ]:
df_parquet = spark.read.parquet("employee_partitioned")

df_parquet.show()
df_parquet.printSchema()

In [ ]:
from pyspark.sql.functions import sum, avg, count

regional_report = df_parquet \
    .groupBy("Region") \
    .agg(
        count("*").alias("Orders"),
        sum("Amount").alias("TotalSales"),
        avg("Amount").alias("AverageSales")
    )

regional_report.show()

In [ ]:
data = [
    (101, "Amit", "Sales", "Mumbai", 2023, 55000, 4.2),
    (102, "Priya", "Sales", "Pune", 2024, 62000, 4.5),
    (103, "Rahul", "IT", "Mumbai", 2022, 75000, 4.1),
    (104, "Sneha", "IT", "Pune", 2023, 82000, 4.7),
    (105, "Karan", "HR", "Delhi", 2024, 50000, 3.9),
    (106, "Neha", "HR", "Mumbai", 2023, 52000, 4.3),
    (107, "Arjun", "Sales", "Delhi", 2022, 68000, 4.6),
    (108, "Pooja", "Finance", "Mumbai", 2024, 72000, 4.4),
    (109, "Rohan", "IT", "Pune", 2024, 90000, 4.8),
    (110, "Meera", "Finance", "Delhi", 2023, 78000, 4.5),
    (111, "Vikram", "HR", "Pune", 2022, 58000, 4.0),
    (112, "Anjali", "IT", "Delhi", 2024, 85000, 4.6)
]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "city",
    "joining_year",
    "salary",
    "rating"
]

df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
df.write \
    .mode("overwrite") \
    .orc("employee_orc")

In [ ]:
orc_df = spark.read.orc("employee_orc")

orc_df.show()

In [ ]:
orc_df.select(
    "employee_name",
    "department",
    "salary"
).show()

In [ ]:
high_salary_df = orc_df.filter(
    orc_df.salary > 70000
)

high_salary_df.show()

In [ ]:
it_high_salary = orc_df.filter(
    (orc_df.department == "IT") &
    (orc_df.salary > 80000)
)

it_high_salary.show()

In [ ]:
from pyspark.sql.functions import avg, max, min, count

department_report = orc_df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("average_salary"),
    max("salary").alias("highest_salary"),
    min("salary").alias("lowest_salary")
)

department_report.show()

In [ ]:
top_performers = orc_df.filter(
    orc_df.rating >= 4.5
)

top_performers.select(
    "employee_name",
    "department",
    "salary",
    "rating"
).show()

In [ ]:
df.write \
    .mode("overwrite") \
    .partitionBy("department") \
    .orc("employee_partitioned_orc")

In [ ]:
partitioned_df = spark.read.orc(
    "employee_partitioned_orc"
)

partitioned_df.show()

In [33]:
it_df = partitioned_df.filter(
    partitioned_df.department == "IT"
)

it_df.show()

+-----------+-------------+------+------------+------+------+----------+
|employee_id|employee_name|  city|joining_year|salary|rating|department|
+-----------+-------------+------+------------+------+------+----------+
|        103|        Rahul|Mumbai|        2022| 75000|   4.1|        IT|
|        112|       Anjali| Delhi|        2024| 85000|   4.6|        IT|
|        109|        Rohan|  Pune|        2024| 90000|   4.8|        IT|
|        104|        Sneha|  Pune|        2023| 82000|   4.7|        IT|
+-----------+-------------+------+------------+------+------+----------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59028)
Traceback (most recent call last):
  File "C:\Program Files\Python314\Lib\socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python314\Lib\socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python314\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python314\Lib\socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "c:\ZensarDemos\Zensar25\Demos\PySpark\Demo1\pysparkenv\Lib\site-packages\pyspark\accumulators.py", line 329, in handle
    poll(accum_updates)
 